BUILDING CHAT BOT WITH LANGCHAIN

In [11]:
!pip install -q llama-index llama-index-llms-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.9/394.9 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.7/162.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 13.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

In [ ]:
import os
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.llms.groq import Groq


In [ ]:
os.environ["GROQ_API_KEY"] = "secrets_ur_api_key"

In [ ]:
llm = Groq(
    model = "llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
def chat():
    history = [
        ChatMessage(role=MessageRole.SYSTEM, content="You are a helpful assistant. Be concise and accurate.")
    ]
    print("LlamaIndex Chatbot: Type 'exit' to quit.\n")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break
        if not user_input:
            continue
        # add user message
        history.append(ChatMessage(role=MessageRole.USER, content=user_input))
        # call LLM with full history
        resp = llm.chat(messages=history)
        answer = resp.message.content
        print(f"Bot: {answer}\n")
        # add assistant message
        history.append(ChatMessage(role=MessageRole.ASSISTANT, content=answer))
        print("-" * 30)

chat()

LlamaIndex Chatbot: Type 'exit' to quit.

You: what is machine learning?
Bot: Machine learning is a subset of artificial intelligence (AI) that involves training algorithms to learn from data and make predictions or decisions without being explicitly programmed. It enables computers to improve their performance on a task over time, based on experience and data.

------------------------------
You: what are subset of machine learning?
Bot: Machine learning has several subsets, including:

1. **Supervised Learning**: Learning from labeled data to make predictions.
2. **Unsupervised Learning**: Discovering patterns in unlabeled data.
3. **Semi-Supervised Learning**: Combining labeled and unlabeled data for learning.
4. **Reinforcement Learning**: Learning through trial and error by interacting with an environment.
5. **Deep Learning**: A subset of machine learning that uses neural networks with multiple layers.

These subsets can be further divided into specific techniques, such as regres

In [12]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.2 MB/s eta 0:00:00


DEPLOYMENT USING STREAMLIT

In [16]:
!pip install streamlit langchain langchain-groq

In [ ]:
%%writefile app.py

import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage

# Page Config
st.set_page_config(
    page_title="Groq Chatbot",
    page_icon="🤖",
    layout="centered"
)

st.title("🤖 AI Chatbot")
st.write("Powered by Groq + LangChain")

# Initialize Session State
if "messages" not in st.session_state:
    st.session_state.messages = []

# Initialize LLM
llm = ChatGroq(
    groq_api_key="ur_secret_api_keys",
    model_name="llama-3.3-70b-versatile",
    temperature=0.3
)

# Display Previous Messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# Chat Input
user_input = st.chat_input("Type your message here...")

if user_input:

    # Store User Message
    st.session_state.messages.append(
        {"role": "user", "content": user_input}
    )

    # Display User Message
    with st.chat_message("user"):
        st.markdown(user_input)

    # Convert Chat History to LangChain Messages
    chat_history = []

    for msg in st.session_state.messages:
        if msg["role"] == "user":
            chat_history.append(HumanMessage(content=msg["content"]))
        else:
            chat_history.append(AIMessage(content=msg["content"]))

    # Get Response
    response = llm.invoke(chat_history)

    bot_response = response.content

    # Store Assistant Response
    st.session_state.messages.append(
        {"role": "assistant", "content": bot_response}
    )

    # Display Assistant Response
    with st.chat_message("assistant"):
        st.markdown(bot_response)

Overwriting app.py


In [19]:
!pip install -q streamlit
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
import subprocess
subprocess.Popen(["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8501"])
!nohup /content/cloudflared-linux-amd64 tunnel --url http://localhost:8501 &

--2026-06-24 10:22:17--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64 [following]
--2026-06-24 10:22:17--  https://github.com/cloudflare/cloudflared/releases/download/2026.6.1/cloudflared-linux-amd64
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/74ac6867-a1bf-4352-b955-c16fbb86d0f6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-24T11%3A09%3A13Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-24T1

In [20]:
!streamlit run /content/app.py &>/content/logs.txt &

In [21]:
!grep -o 'https://.*\.trycloudflare.com' nohup.out | head -n 1 | xargs -I {} echo "Your tunnel url {}"

Your tunnel url https://pills-accessed-performed-knowledge.trycloudflare.com
